# 5.1 🛡️ Blindando el Dominio: Validación y Modelos Confiables

En un sistema real **no se confía en el input**.

En analítica y BI esto aparece todos los días:

- Tickers vacíos (`""`) o con espacios (`" AAPL "`)
- Cantidades negativas o cero
- Precios con `None`, `NaN` o valores absurdos
- JSON externos incompletos o con tipos incorrectos
- Fechas con formatos inconsistentes

Si tu dominio no valida, tu sistema termina con:

- Bugs silenciosos
- Resultados financieros incoherentes
- Tests frágiles
- Pipelines rotos
- Problemas de seguridad (inyección / abuso de endpoints)

---

## 🎯 Objetivo

Al final de esta sección podrás:

1. Definir **invariantes del dominio** (reglas que siempre deben cumplirse).
2. Implementar **Value Objects** para evitar “primitivos sueltos”.
3. Usar **excepciones de dominio** (errores explícitos, no `None` silenciosos).
4. Aplicar validación con **Pydantic** (mentalidad backend) para parsing/serialización.

---

## 🧭 Idea central: El dominio protege al sistema

```{mermaid}
flowchart LR
    A[Input externo: usuario / API / CSV] --> B[Validación estricta]
    B --> C[Dominio: Instrumento / Posición / Portafolio]
    C --> D[Analítica: indicadores / Markowitz / ML]
    D --> E[Salida: reporte / recomendación / API]
```


## 1) ¿Qué es una invariante?

Una **invariante** es una condición que **siempre** debe ser cierta para que el dominio sea coherente.

Ejemplos para SmartPortfolio:

- `Ticker` no vacío, sin espacios, en mayúsculas
- `Cantidad` > 0
- `Precio` ≥ 0
- `Días futuros` > 0 (no se predice al pasado)
- `Moneda` pertenece a un conjunto válido (opcional)

Si rompes invariantes, el dominio entra en estados inválidos y luego “nadie sabe” por qué falla.

---

## 2) Antipatrón: Primitive Obsession

Cuando el dominio usa `str`, `int`, `float` para todo, se vuelve fácil pasar valores inválidos:

- `ticker="   "`
- `cantidad=-10`
- `precio=float("nan")`

Solución: **Value Objects**.


## 3) Value Objects: tipos pequeños con reglas grandes

Un Value Object:

- Encapsula reglas (validación)
- Hace el código auto-documentado
- Previene errores por construcción

```{mermaid}
flowchart TD
    A[str/int/float sueltos] -->|riesgo| B[estados inválidos]
    C[Value Objects] -->|protección| D[estados válidos]
```


In [1]:
from __future__ import annotations

from dataclasses import dataclass
import math
import re


class DomainError(ValueError):
    """Base exception for domain rule violations."""


class InvalidTickerError(DomainError):
    """Raised when a ticker is empty/invalid."""


class InvalidQuantityError(DomainError):
    """Raised when a quantity is not strictly positive."""


class InvalidPriceError(DomainError):
    """Raised when a price is negative, NaN, or not finite."""


_TICKER_RE = re.compile(r"^[A-Z0-9.\-]+$")


@dataclass(frozen=True, slots=True)
class Ticker:
    """
    Ticker symbol value object.

    Rules:
    - Must be a non-empty string after stripping.
    - Stored in uppercase.
    - Must match allowed chars: A-Z, 0-9, '.', '-'
    """
    value: str

    def __post_init__(self) -> None:
        raw = (self.value or "").strip()
        if not raw:
            raise InvalidTickerError("Ticker must be a non-empty string.")

        norm = raw.upper()
        if not _TICKER_RE.match(norm):
            raise InvalidTickerError(f"Ticker contains invalid characters: {raw!r}")

        object.__setattr__(self, "value", norm)


@dataclass(frozen=True, slots=True)
class Quantity:
    """
    Quantity value object.

    Rules:
    - Must be int (bool not allowed).
    - Must be strictly positive.
    """
    value: int

    def __post_init__(self) -> None:
        if isinstance(self.value, bool) or not isinstance(self.value, int):
            raise InvalidQuantityError("Quantity must be an integer.")
        if self.value <= 0:
            raise InvalidQuantityError("Quantity must be > 0.")


@dataclass(frozen=True, slots=True)
class Money:
    """
    Money value object (simple numeric wrapper).

    Rules:
    - Must be finite number.
    - Must be >= 0.
    """
    value: float

    def __post_init__(self) -> None:
        try:
            v = float(self.value)
        except (TypeError, ValueError) as exc:
            raise InvalidPriceError("Money must be numeric.") from exc

        if not math.isfinite(v):
            raise InvalidPriceError("Money must be finite (not NaN/inf).")
        if v < 0:
            raise InvalidPriceError("Money must be >= 0.")

        object.__setattr__(self, "value", v)

### ✅ ¿Qué ganamos con Value Objects?

- Los errores aparecen **cerca de donde nacen**
- El dominio queda “tipado” (aunque sea Python)
- La intención es clara (`Ticker("AAPL")` vs `"AAPL"`)
- Los tests se vuelven más simples

---

## 4) Dominio mínimo usando Value Objects

Ahora construimos objetos del dominio que **solo aceptan estados válidos**.

```{mermaid}
classDiagram
    class Instrumento {
        +Ticker ticker
        +str nombre
    }

    class Posicion {
        +Instrumento instrumento
        +Quantity cantidad
        +Money precio_entrada
        +pnl_no_realizado(precio_actual)
    }

    Instrumento "1" --> "many" Posicion
```


In [2]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True, slots=True)
class Instrumento:
    """Minimal domain entity."""
    ticker: Ticker
    nombre: str


@dataclass(frozen=True, slots=True)
class Posicion:
    """Domain entity representing a holding."""
    instrumento: Instrumento
    cantidad: Quantity
    precio_entrada: Money

    def pnl_no_realizado(self, precio_actual: Money) -> float:
        """
        Compute unrealized PnL (profit and loss) for the position.

        PnL = (precio_actual - precio_entrada) * cantidad
        """
        return (precio_actual.value - self.precio_entrada.value) * self.cantidad.value


def main() -> None:
    """Small smoke-test / usage example."""
    inst = Instrumento(ticker=Ticker("TSLA"), nombre="Tesla Inc.")
    pos = Posicion(
        instrumento=inst,
        cantidad=Quantity(3),
        precio_entrada=Money(200.0),
    )

    pnl: float = pos.pnl_no_realizado(Money(250.0))
    print("PnL:", pnl)


if __name__ == "__main__":
    main()

PnL: 150.0


## 5) Pydantic: validación + parsing + serialización (mentalidad backend)

En backend moderno, normalmente se usa un “schema” para validar inputs externos.

Eso es exactamente lo que hace **Pydantic**:

- Convierte tipos (`"10"` → `10` cuando aplica)
- Aplica reglas (`min_length`, `gt`, etc.)
- Serializa a JSON de forma segura

### Instalación (con Poetry)

```bash
poetry add pydantic
```

> Nota: El curso puede usar Pydantic v1 o v2. El ejemplo de abajo funciona en ambos.


### ¿Dónde usar Pydantic en SmartPortfolio?

- Entrada CLI / API (DTOs)
- Configuración (Settings)
- Parsing de respuestas JSON
- Serialización (guardar portafolio a JSON)

Conceptualmente:

```{mermaid}
flowchart LR
    A[Input externo] --> B[Pydantic DTO]
    B --> C[Value Objects]
    C --> D[Dominio]
```


In [3]:
from __future__ import annotations

from typing import Optional

try:
    # Pydantic v2
    from pydantic import BaseModel, Field, ValidationError, field_validator
    PYDANTIC_V2 = True
except ImportError:  # pragma: no cover
    # Pydantic v1
    from pydantic import BaseModel, Field, ValidationError, validator
    PYDANTIC_V2 = False


class OrdenCompraDTO(BaseModel):
    """
    Validated external input schema (DTO).

    This model represents untrusted external input.
    It should be converted into domain objects (Value Objects)
    before entering the core domain.
    """

    ticker: str = Field(
        min_length=1,
        description="Ticker symbol (e.g., AAPL)",
    )
    cantidad: int = Field(
        gt=0,
        description="Number of shares (> 0)",
    )
    precio_entrada: float = Field(
        ge=0,
        description="Entry price (>= 0)",
    )
    nombre: Optional[str] = Field(
        default=None,
        description="Optional display name",
    )

    # --- Normalization logic ---
    if PYDANTIC_V2:

        @field_validator("ticker")
        @classmethod
        def normalize_ticker(cls, value: str) -> str:
            return value.strip().upper()

    else:

        @validator("ticker")
        def normalize_ticker(cls, value: str) -> str:
            return value.strip().upper()


# Example usage
dto = OrdenCompraDTO(
    ticker=" aapl ",
    cantidad=10,
    precio_entrada=150.5,
    nombre="Apple",
)

# v2 uses model_dump(), v1 uses dict()
dump = dto.model_dump() if hasattr(dto, "model_dump") else dto.dict()

print(dump)

{'ticker': 'AAPL', 'cantidad': 10, 'precio_entrada': 150.5, 'nombre': 'Apple'}


### Convertir DTO → Dominio (mapeo explícito)

Muy importante: **no mezcles** DTO con dominio.

- DTO: admite “ensuciamiento” y conversión
- Dominio: debe estar limpio y coherente

```{mermaid}
sequenceDiagram
    participant UI as CLI/API
    participant DTO as Pydantic DTO
    participant DOM as Dominio
    UI->>DTO: input raw
    DTO-->>DOM: map to Value Objects
    DOM-->>UI: result / error
```


In [4]:
def dto_a_posicion(dto: OrdenCompraDTO) -> Posicion:
    """
    Map a validated DTO into a domain object.

    Important:
    - DTO is already validated structurally (types, ranges).
    - Domain objects enforce business invariants.
    """

    # Create Value Objects explicitly
    ticker_vo: Ticker = Ticker(dto.ticker)
    cantidad_vo: Quantity = Quantity(dto.cantidad)
    precio_vo: Money = Money(dto.precio_entrada)

    instrumento: Instrumento = Instrumento(
        ticker=ticker_vo,
        nombre=dto.nombre or ticker_vo.value,
    )

    return Posicion(
        instrumento=instrumento,
        cantidad=cantidad_vo,
        precio_entrada=precio_vo,
    )


# Example usage
pos2: Posicion = dto_a_posicion(dto)

print(
    pos2.instrumento.ticker.value,
    pos2.cantidad.value,
    pos2.precio_entrada.value,
)

AAPL 10 150.5


## 6) Manejo explícito de errores

En BI/DS es común “arreglar” datos con `try/except` genéricos o `fillna` indiscriminado.

En sistemas, preferimos:

- **Fail fast**: fallar temprano
- **Errores claros**: con causa y contexto
- No producir resultados silenciosamente incorrectos


In [5]:
from __future__ import annotations

from typing import Any, Dict


def _short_pydantic_error(err: ValidationError) -> str:
    """
    Extract a short, readable message from Pydantic ValidationError.
    Works for both Pydantic v1 and v2.
    """
    try:
        # v2
        errors = err.errors()
        first = errors[0]
        loc = ".".join(str(x) for x in first.get("loc", []))
        msg = first.get("msg", str(err))
        return f"{loc}: {msg}" if loc else msg
    except Exception:
        # fallback
        return str(err).splitlines()[0]


bad_inputs: list[Dict[str, Any]] = [
    {"ticker": "", "cantidad": 1, "precio_entrada": 10.0},
    {"ticker": "MSFT", "cantidad": 0, "precio_entrada": 10.0},
    {"ticker": "MSFT", "cantidad": 1, "precio_entrada": -1.0},
    {"ticker": "   ", "cantidad": 5, "precio_entrada": 99.0},
]

for payload in bad_inputs:
    try:
        dto_bad = OrdenCompraDTO(**payload)
        _ = dto_a_posicion(dto_bad)
        print(f"✅ OK: {payload}")
    except ValidationError as e:
        print(f"🟠 DTO validation error: {payload}")
        print("   ↳", _short_pydantic_error(e))
    except DomainError as e:
        print(f"🔴 Domain error: {payload}")
        print("   ↳", str(e))

🟠 DTO validation error: {'ticker': '', 'cantidad': 1, 'precio_entrada': 10.0}
   ↳ ticker: String should have at least 1 character
🟠 DTO validation error: {'ticker': 'MSFT', 'cantidad': 0, 'precio_entrada': 10.0}
   ↳ cantidad: Input should be greater than 0
🟠 DTO validation error: {'ticker': 'MSFT', 'cantidad': 1, 'precio_entrada': -1.0}
   ↳ precio_entrada: Input should be greater than or equal to 0
🔴 Domain error: {'ticker': '   ', 'cantidad': 5, 'precio_entrada': 99.0}
   ↳ Ticker must be a non-empty string.


## 7) Mini‑Cheatsheet (para aplicar en SmartPortfolio)

### ✅ Reglas recomendadas (mínimas)

- `Ticker`: strip + uppercase + no vacío
- `Cantidad`: `int` y `> 0`
- `Precio`: `float` finito y `>= 0`
- Excepciones propias (DomainError)
- DTOs (Pydantic) en la frontera del sistema (input/output)
- Value Objects dentro del dominio

### 🧠 Patrón mental

> DTO valida “lo que llega”  
> Dominio valida “lo que existe”

---

## 🎯 Cierre

En esta sección movimos el sistema de:

- “Funciona si el usuario se porta bien”

a

- “Funciona **porque** el dominio se protege”

En 5.2 vamos a sacar secretos del código (configuración segura).  
En 5.3 vamos a acelerar descargas masivas (async).
